# Day 18 - Keras Optimizer Comparison + ReduceLROnPlateau


## 학습 목표
- Adam, RMSprop, SGD 등 Optimizer 비교
- ReduceLROnPlateau로 학습률 동적 조정


## 1. 데이터 준비


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

data = load_diabetes()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_te = scaler.transform(X_te)
print("Train:", X_tr.shape)


## 2. Optimizer 비교 실험


In [ ]:
def build_and_train(optimizer, name, epochs=80):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(X_tr.shape[1],)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer=optimizer, loss='mse')
    hist = model.fit(X_tr, y_tr, epochs=epochs, batch_size=32, validation_split=0.2, verbose=0)
    final_val = hist.history['val_loss'][-1]
    print(f"[{name:12s}] final val_loss = {final_val:.4f}")
    return hist

hist_adam = build_and_train(tf.keras.optimizers.Adam(0.001), "Adam")
hist_rms = build_and_train(tf.keras.optimizers.RMSprop(0.001), "RMSprop")
hist_sgd = build_and_train(tf.keras.optimizers.SGD(0.01), "SGD")
hist_sgdm = build_and_train(tf.keras.optimizers.SGD(0.01, momentum=0.9), "SGD+Momentum")


## 3. 학습 곡선 비교


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(hist_adam.history['val_loss'], label='Adam')
plt.plot(hist_rms.history['val_loss'], label='RMSprop')
plt.plot(hist_sgd.history['val_loss'], label='SGD')
plt.plot(hist_sgdm.history['val_loss'], label='SGD+Momentum')
plt.legend()
plt.title('Optimizer Comparison (val_loss)')
plt.xlabel('Epoch')
plt.ylabel('Val Loss')
plt.grid(True, alpha=0.3)
plt.show()


## 4. ReduceLROnPlateau 적용


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X_tr.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)
])
model.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss='mse')  # 의도적으로 큰 LR

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=8,
    min_lr=1e-6,
    verbose=1
)
early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

hist_rlr = model.fit(
    X_tr, y_tr,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[reduce_lr, early],
    verbose=0
)

print(f"Final val_loss: {hist_rlr.history['val_loss'][-1]:.4f}")
print(f"Final LR      : {float(tf.keras.backend.get_value(model.optimizer.learning_rate)):.6f}")

plt.figure(figsize=(10, 4))
plt.plot(hist_rlr.history['loss'], label='train')
plt.plot(hist_rlr.history['val_loss'], label='val')
plt.legend()
plt.title('ReduceLROnPlateau Training Curve')
plt.xlabel('Epoch')
plt.grid(True, alpha=0.3)
plt.show()
